# TP3 — KPIs par canal & campagne — Lumina & Co

Le CMO reçoit chaque semaine des rapports contradictoires de ses différents canaux marketing. Objectif de ce notebook : lui donner des **chiffres clairs par canal** (CAC, CPA, ROAS, taux de conversion) et une **première lecture critique** de ce qu'ils veulent vraiment dire — avant de comparer les modèles d'attribution dans le second notebook (`TP3 - Attribution Multicanal`).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_DIR = "Lumina & Co - CRM"


## Chargement et vue d'ensemble

In [2]:
campaigns = pd.read_csv(f"{DATA_DIR}/campaigns.csv")
touchpoints = pd.read_csv(f"{DATA_DIR}/touchpoints.csv")
touchpoints['timestamp'] = pd.to_datetime(touchpoints['timestamp'])

print("campaigns.csv   :", campaigns.shape)
print("touchpoints.csv :", touchpoints.shape)
campaigns


campaigns.csv   : (6, 14)
touchpoints.csv : (1080897, 13)


,campaign_name,n_touchpoints,n_clients,n_clicks,n_conversions,total_cost,revenue,ctr,conversion_rate,cpa,roas,primary_channel,start_date,end_date
0,Back_to_School,171006,25527,85710,13192,"159,256.41","725,541.50",0.50,0.52,12.07,4.56,social,2025-08-09 23:22:22.554461652,2025-10-15 23:59:59.624507864
1,Black_Friday,193299,26937,101841,18777,"205,749.41","957,170.87",0.53,0.70,10.96,4.65,display,2025-10-02 18:51:37.807091349,2025-12-27 23:59:37.789190263
2,Spring_Launch,195451,26612,103868,20372,"374,813.33","1,686,555.27",0.53,0.77,18.40,4.50,display,2026-04-02 18:02:14.505881373,2026-06-29 23:59:25.975461263
3,Summer_Sale,170209,25378,84734,12714,"226,774.16","621,009.60",0.50,0.50,17.84,2.74,display,2025-06-16 05:36:28.494713012,2025-08-22 23:58:58.285941573
4,Valentine,176986,25850,90122,15141,"247,465.96","853,184.79",0.51,0.59,16.34,3.45,display,2026-02-07 14:52:39.479203239,2026-04-14 23:58:57.777460163
5,Winter_Promo,173946,25638,88093,14170,"133,949.00","716,632.64",0.51,0.55,9.45,5.35,display,2025-12-14 23:13:04.314410780,2026-02-19 23:59:53.504774786


In [3]:
touchpoints.head()


,touchpoint_id,customer_id,invoice_id,journey_id,campaign_name,timestamp,channel,position,n_touchpoints,click,converted,cost,is_last_before_conversion
0,0,12346,NaN,conv_541431,Summer_Sale,2025-07-27 22:03:12.473059457,email,2,4,1,0,0.02,0
1,1,12346,NaN,conv_541431,Summer_Sale,2025-08-01 18:48:15.521925754,affiliate,3,4,1,0,0.00,0
2,2,12346,"541,431.00",conv_541431,Summer_Sale,2025-08-08 19:43:30.750000041,retargeting,4,4,1,1,0.01,1
3,3,12346,NaN,exp_3209,Back_to_School,2025-08-24 04:38:02.208965716,social,1,8,0,0,0.00,0
4,4,12346,NaN,exp_3209,Back_to_School,2025-08-30 13:11:10.761750471,social,2,8,1,0,2.04,0


### Qualité des données

`touchpoint_id` est-il bien unique ? `customer_id` est-il toujours renseigné (contrairement à `transactions.csv` au Jour 1) ? `invoice_id` n'est renseigné que lorsqu'il y a conversion — c'est cohérent avec sa fonction (relier le point de contact à l'achat qu'il a généré).


In [4]:
print("touchpoint_id dupliqué :", touchpoints['touchpoint_id'].duplicated().sum())
print("customer_id manquant      :", touchpoints['customer_id'].isna().sum())
print("invoice_id renseigné      :", touchpoints['invoice_id'].notna().sum(), "/", len(touchpoints))
print("invoice_id renseigné ssi converted==1 ?",
      (touchpoints.loc[touchpoints['converted']==1, 'invoice_id'].notna().all()
       and touchpoints.loc[touchpoints['converted']==0, 'invoice_id'].isna().all()))
print("\nCanaux :", touchpoints['channel'].unique())
print("\nRépartition par canal (nombre de points de contact) :")
print(touchpoints['channel'].value_counts())


touchpoint_id dupliqué : 0
customer_id manquant      : 0
invoice_id renseigné      : 94366 / 1080897
invoice_id renseigné ssi converted==1 ? True

Canaux : ['email' 'affiliate' 'retargeting' 'social' 'display' 'direct'
 'search_paid']

Répartition par canal (nombre de points de contact) :
channel
display        316530
social         315763
retargeting    158102
email          150733
affiliate       54516
direct          43919
search_paid     41334
Name: count, dtype: int64


### Réconciliation `campaigns.csv` vs `touchpoints.csv`

Contrairement à `customers.csv`/`transactions.csv` au Jour 1 (qui ne se réconciliaient pas), on vérifie ici si l'agrégat `campaigns.csv` correspond bien à ce qu'on recalcule depuis le détail `touchpoints.csv`.


In [5]:
recalc = touchpoints.groupby('campaign_name').agg(
    n_touchpoints_recalc=('touchpoint_id', 'count'),
    n_conversions_recalc=('converted', 'sum'),
    total_cost_recalc=('cost', 'sum'),
).reset_index()

check = campaigns[['campaign_name','n_touchpoints','n_conversions','total_cost']].merge(recalc, on='campaign_name')
check['ecart_touchpoints'] = check['n_touchpoints'] - check['n_touchpoints_recalc']
check['ecart_conversions'] = check['n_conversions'] - check['n_conversions_recalc']
check['ecart_cost'] = (check['total_cost'] - check['total_cost_recalc']).abs()
check


,campaign_name,n_touchpoints,n_conversions,total_cost,n_touchpoints_recalc,n_conversions_recalc,total_cost_recalc,ecart_touchpoints,ecart_conversions,ecart_cost
0,Back_to_School,171006,13192,"159,256.41",171006,13192,"159,256.41",0,0,0.00
1,Black_Friday,193299,18777,"205,749.41",193299,18777,"205,749.41",0,0,0.00
2,Spring_Launch,195451,20372,"374,813.33",195451,20372,"374,813.33",0,0,0.00
3,Summer_Sale,170209,12714,"226,774.16",170209,12714,"226,774.16",0,0,0.00
4,Valentine,176986,15141,"247,465.96",176986,15141,"247,465.96",0,0,0.00
5,Winter_Promo,173946,14170,"133,949.00",173946,14170,"133,949.00",0,0,0.00


**Contrairement au Jour 1, les deux fichiers se réconcilient parfaitement** (écarts nuls sur les 6 campagnes). `campaigns.csv` est donc un agrégat fiable de `touchpoints.csv` — on peut s'appuyer dessus pour vérifier nos propres calculs au fil du notebook.


## Étape 1 — KPIs par canal

### Comprendre la colonne `converted`

Avant de calculer quoi que ce soit, un point méthodologique important : la colonne `converted` ne s'active que sur le **dernier point de contact avant l'achat** d'un parcours qui a effectivement converti (on le vérifie ci-dessous). Cela veut dire que **toute agrégation directe sur `converted` est déjà, de fait, un modèle d'attribution "dernier contact" (last touch)** — sans qu'on l'ait choisi explicitement. On le garde en tête : c'est précisément le sujet du second notebook.


In [6]:
touchpoints['journey_prefix'] = touchpoints['journey_id'].str.split('_').str[0]

conv_journeys = touchpoints[touchpoints['journey_prefix'] == 'conv']
last_rows = conv_journeys.loc[conv_journeys.groupby('journey_id')['position'].idxmax()]

print(f"Parcours 'conv_*' (ayant converti) : {conv_journeys['journey_id'].nunique():,}")
print(f"Parcours 'exp_*' (jamais convertis) : {touchpoints[touchpoints['journey_prefix']=='exp']['journey_id'].nunique():,}")
print(f"La dernière ligne d'un parcours 'conv_*' a bien converted==1 dans "
      f"{(last_rows['converted']==1).mean()*100:.1f}% des cas")
print(f"Une ligne 'converted'==1 apparaît-elle dans un parcours 'exp_*' ? "
      f"{(touchpoints[touchpoints['journey_prefix']=='exp']['converted']==1).sum()} fois (jamais, comme attendu)")


Parcours 'conv_*' (ayant converti) : 94,366
Parcours 'exp_*' (jamais convertis) : 120,531
La dernière ligne d'un parcours 'conv_*' a bien converted==1 dans 100.0% des cas


Une ligne 'converted'==1 apparaît-elle dans un parcours 'exp_*' ? 0 fois (jamais, comme attendu)


### Construction du tableau de base par canal

In [7]:
channel_kpi = touchpoints.groupby('channel').agg(
    n_touchpoints=('touchpoint_id', 'count'),
    n_clients=('customer_id', 'nunique'),
    n_clicks=('click', 'sum'),
    n_conversions=('converted', 'sum'),
    total_cost=('cost', 'sum'),
).reset_index()

channel_kpi['ctr'] = channel_kpi['n_clicks'] / channel_kpi['n_touchpoints']
channel_kpi['conversion_rate'] = channel_kpi['n_conversions'] / channel_kpi['n_clients']
channel_kpi['cpa'] = channel_kpi['total_cost'] / channel_kpi['n_conversions']

channel_kpi.sort_values('total_cost', ascending=False)


,channel,n_touchpoints,n_clients,n_clicks,n_conversions,total_cost,ctr,conversion_rate,cpa
0,affiliate,54516,20800,44884,26979,"1,128,776.74",0.82,1.30,41.84
6,social,315763,47500,142089,0,"166,324.04",0.45,0.00,inf
5,search_paid,41334,17513,31548,13666,"47,264.48",0.76,0.78,3.46
3,email,150733,43141,75666,14355,"2,263.28",0.50,0.33,0.16
4,retargeting,158102,43482,83548,22563,"1,897.65",0.53,0.52,0.08
2,display,316530,47555,142192,0,"1,482.10",0.45,0.00,inf
1,direct,43919,18232,34441,16803,0.00,0.78,0.92,0.00


*(Vérification silencieuse : ces formules de `ctr`, `conversion_rate` et `cpa` reproduisent exactement les colonnes déjà présentes dans `campaigns.csv` au niveau campagne — on les réutilise donc en confiance au niveau canal.)*

### Revenu et ROAS par canal

Le revenu généré par un canal = la somme des montants des factures (`transactions.csv`) liées aux conversions dont ce canal a été le dernier point de contact.


In [8]:
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv", usecols=['invoice_id','quantity','unit_price'])
transactions['line_total'] = transactions['quantity'] * transactions['unit_price']
invoice_total = transactions.groupby('invoice_id')['line_total'].sum()

conversions = touchpoints[touchpoints['converted'] == 1].copy()
# invoice_id est stocké en float côté touchpoints (ex: 541431.0) et en texte côté transactions (ex: "541431") -> on aligne les deux
conversions['invoice_id_str'] = conversions['invoice_id'].astype(int).astype(str)
conversions['revenue'] = conversions['invoice_id_str'].map(invoice_total)

revenue_by_channel = conversions.groupby('channel')['revenue'].sum().rename('revenue')
channel_kpi = channel_kpi.merge(revenue_by_channel, on='channel', how='left').fillna({'revenue': 0})
channel_kpi['roas'] = channel_kpi['revenue'] / channel_kpi['total_cost']

channel_kpi[['channel','n_touchpoints','total_cost','n_conversions','revenue','roas','cpa','conversion_rate','ctr']].sort_values('roas', ascending=False)


,channel,n_touchpoints,total_cost,n_conversions,revenue,roas,cpa,conversion_rate,ctr
1,direct,43919,0.00,16803,"3,204,574.65",inf,0.00,0.92,0.78
4,retargeting,158102,"1,897.65",22563,"4,455,952.61","2,348.15",0.08,0.52,0.53
3,email,150733,"2,263.28",14355,"2,603,351.06","1,150.26",0.16,0.33,0.50
5,search_paid,41334,"47,264.48",13666,"2,824,693.39",59.76,3.46,0.78,0.76
0,affiliate,54516,"1,128,776.74",26979,"5,313,106.12",4.71,41.84,1.30,0.82
2,display,316530,"1,482.10",0,0.00,0.00,inf,0.00,0.45
6,social,315763,"166,324.04",0,0.00,0.00,inf,0.00,0.45


### CAC — repérer les primo-achats

> **CAC ≠ CPA.** Le CAC ne compte que les nouveaux clients, le CPA compte toute conversion, y compris le réachat d'un client déjà fidèle.

**Méthode :** pour chaque conversion, on récupère la date de la facture (`invoice_date` dans `transactions.csv`) et on la compare à `first_purchase` du client dans `customers.csv`. Si elles sont égales, c'est un primo-achat.


In [9]:
invoice_dates = pd.read_csv(f"{DATA_DIR}/transactions.csv", usecols=['invoice_id','invoice_date'])
invoice_dates['invoice_date'] = pd.to_datetime(invoice_dates['invoice_date']).dt.normalize()
invoice_date_by_id = invoice_dates.groupby('invoice_id')['invoice_date'].first()

customers = pd.read_csv(f"{DATA_DIR}/customers.csv", usecols=['customer_id','first_purchase'])
customers['first_purchase'] = pd.to_datetime(customers['first_purchase']).dt.normalize()

conversions['invoice_date'] = conversions['invoice_id_str'].map(invoice_date_by_id)
conversions = conversions.merge(customers, on='customer_id', how='left')
conversions['is_primo_achat'] = conversions['invoice_date'] == conversions['first_purchase']

print(f"Primo-achats parmi les conversions trackées : {conversions['is_primo_achat'].sum():,} / {len(conversions):,} "
      f"({conversions['is_primo_achat'].mean()*100:.1f} %)")

primo_by_channel = conversions.groupby('channel')['is_primo_achat'].sum().rename('n_primo_achats')
channel_kpi = channel_kpi.merge(primo_by_channel, on='channel', how='left').fillna({'n_primo_achats': 0})
channel_kpi['cac'] = channel_kpi['total_cost'] / channel_kpi['n_primo_achats']

cols = ['channel','n_touchpoints','total_cost','n_conversions','n_primo_achats','revenue','roas','cpa','cac','conversion_rate','ctr']
channel_kpi[cols].sort_values('total_cost', ascending=False)


Primo-achats parmi les conversions trackées : 1,749 / 94,366 (1.9 %)


,channel,n_touchpoints,total_cost,n_conversions,n_primo_achats,revenue,roas,cpa,cac,conversion_rate,ctr
0,affiliate,54516,"1,128,776.74",26979,507.00,"5,313,106.12",4.71,41.84,"2,226.38",1.30,0.82
6,social,315763,"166,324.04",0,0.00,0.00,0.00,inf,inf,0.00,0.45
5,search_paid,41334,"47,264.48",13666,266.00,"2,824,693.39",59.76,3.46,177.69,0.78,0.76
3,email,150733,"2,263.28",14355,258.00,"2,603,351.06","1,150.26",0.16,8.77,0.33,0.50
4,retargeting,158102,"1,897.65",22563,441.00,"4,455,952.61","2,348.15",0.08,4.30,0.52,0.53
2,display,316530,"1,482.10",0,0.00,0.00,0.00,inf,inf,0.00,0.45
1,direct,43919,0.00,16803,277.00,"3,204,574.65",inf,0.00,0.00,0.92,0.78


**Seulement 1,9 % des conversions trackées sont des primo-achats** (les 98 % restants sont des réachats de clients déjà connus). Deux conséquences importantes :
1. Le **CAC est presque partout bien plus élevé que le CPA** — cohérent avec l'avertissement de l'énoncé, puisqu'il porte sur un sous-ensemble beaucoup plus restreint (250 à 510 primo-achats par canal, contre 13 000 à 27 000 conversions totales).
2. Avec un échantillon aussi réduit par canal, **le CAC individuel de chaque canal est statistiquement fragile** — une poignée de gros/petits paniers suffit à le faire bouger fortement. À interpréter comme un ordre de grandeur, pas une valeur de pilotage fine.


### Classement des canaux selon chaque métrique

Le classement est-il le même selon la métrique choisie ?


In [10]:
for metric, ascending in [('roas', False), ('cpa', True), ('cac', True), ('conversion_rate', False)]:
    ordered = channel_kpi.sort_values(metric, ascending=ascending)['channel'].tolist()
    print(f"{metric:>16} : {ordered}")


            roas : ['direct', 'retargeting', 'email', 'search_paid', 'affiliate', 'display', 'social']
             cpa : ['direct', 'retargeting', 'email', 'search_paid', 'affiliate', 'display', 'social']
             cac : ['direct', 'retargeting', 'email', 'search_paid', 'affiliate', 'display', 'social']
 conversion_rate : ['affiliate', 'direct', 'search_paid', 'retargeting', 'email', 'display', 'social']


**Non, le classement change radicalement selon la métrique** :

- Sur le **volume brut** (nombre de points de contact, de clics), `display` et `social` dominent largement.
- Sur le **ROAS**, ce sont `retargeting` et `email` qui explosent tout (ROAS de plusieurs milliers), simplement parce que leur coût est quasiment nul (canaux "propriétaires" — un email ou un pixel de retargeting coûte presque rien à déclencher) alors qu'ils récoltent le crédit de conversions dont le vrai travail de persuasion a souvent été fait plus tôt dans le parcours par un canal payant. **Un ROAS énorme sur un canal quasi-gratuit n'est pas forcément un signal de performance — c'est aussi un artefact du dernier-clic.**
- Sur le **CPA/CAC**, `display` et `social` apparaissent avec un coût par acquisition **infini** : `n_conversions = 0` pour ces deux canaux (voir ci-dessous). Ce n'est pas qu'ils ne servent à rien — c'est que la colonne `converted` (= dernier contact) ne leur attribue jamais aucun crédit.


In [11]:
print("Conversions attribuées (dernier contact) par canal :")
print(channel_kpi.set_index('channel')['n_conversions'].sort_values(ascending=False))


Conversions attribuées (dernier contact) par canal :
channel
affiliate      26979
retargeting    22563
direct         16803
email          14355
search_paid    13666
display            0
social             0
Name: n_conversions, dtype: int64


**`display` et `social` ont 0 conversion attribuée alors qu'ils reçoivent, à eux deux, plus de 630 000 points de contact** (58 % du volume total). Un problème méthodologique, pas un problème produit — creusé en détail dans le notebook d'attribution.


### Vanity metrics

> Parmi les colonnes de `touchpoints.csv`, lesquelles seraient de simples métriques de vanité si on les regardait isolément ?

- **`n_touchpoints`** (nombre de points de contact) et **`n_clicks`** sont les candidats évidents : `display` a le plus grand volume des deux, et pourtant 0 conversion mesurée. Un gros chiffre de volume ne dit rien, seul, sur l'efficacité commerciale.
- Elles deviennent de vrais KPI **une fois rapportées à un coût et/ou une conversion** : `n_clicks / n_touchpoints` (CTR) dit si le message capte l'attention, `n_conversions / total_cost` (CPA) dit si ça se traduit en ventes, `revenue / total_cost` (ROAS) dit si c'est rentable. Ce n'est jamais le chiffre brut qui est vain, c'est de le lire **sans son dénominateur**.


### Pour aller plus loin (optionnel) — CLTV et ratio LTV:CAC

**Méthode simplifiée :** `CLTV = rythme de dépense annualisé × durée de vie estimée × marge assumée`.

**Hypothèses documentées** (faute de barème fourni par l'entreprise) : durée de vie estimée = **2 ans**, marge assumée = **35 %** (ordre de grandeur usuel en cosmétique DTC). Le rythme de dépense annualisé est estimé, pour chaque client primo-acquis par un canal, à partir de son historique connu dans `customers.csv` (`total_spent` rapporté à son ancienneté en jours, ramené sur un an, avec un plancher de 30 jours pour éviter qu'un client tout juste acquis ne produise un taux extrême).


In [12]:
LIFETIME_YEARS = 2
ASSUMED_MARGIN = 0.35

customers_full = pd.read_csv(f"{DATA_DIR}/customers.csv")
customers_full['first_purchase'] = pd.to_datetime(customers_full['first_purchase'])
snapshot_date = pd.to_datetime(touchpoints['timestamp']).max()

primo_customers = conversions.loc[conversions['is_primo_achat'], ['channel','customer_id']].drop_duplicates()
primo_customers = primo_customers.merge(customers_full[['customer_id','total_spent','first_purchase']], on='customer_id', how='left')

primo_customers['age_days'] = (snapshot_date - primo_customers['first_purchase']).dt.days.clip(lower=30)
primo_customers['annualized_spend'] = primo_customers['total_spent'] / primo_customers['age_days'] * 365
primo_customers['cltv'] = primo_customers['annualized_spend'] * LIFETIME_YEARS * ASSUMED_MARGIN

cltv_by_channel = primo_customers.groupby('channel').agg(
    n_primo_customers=('customer_id', 'count'),
    cltv_moyenne=('cltv', 'mean'),
).reset_index()

cltv_table = channel_kpi[['channel','cac']].merge(cltv_by_channel, on='channel', how='right')
cltv_table['ltv_cac'] = cltv_table['cltv_moyenne'] / cltv_table['cac']
cltv_table.sort_values('ltv_cac', ascending=False)


,channel,cac,n_primo_customers,cltv_moyenne,ltv_cac
1,direct,0.00,274,"3,969.60",inf
3,retargeting,4.30,435,"5,847.74","1,358.98"
2,email,8.77,257,"3,522.37",401.53
4,search_paid,177.69,266,"6,104.57",34.36
0,affiliate,"2,226.38",501,"7,073.39",3.18


Un canal avec un bon CAC (`retargeting`, `email`) n'est ici pas directement comparable à `affiliate` sur le seul critère du CAC — mais **le ratio LTV:CAC confirme et amplifie l'écart** déjà visible plus haut : les canaux au coût d'acquisition dérisoire (`retargeting`, `email`) affichent un LTV:CAC excellent, tandis qu'`affiliate` — avec un CAC de plus de 2 000 € pour un client dont la valeur vie estimée est de quelques centaines d'euros — apparaît structurellement déficitaire **s'il est jugé uniquement comme canal d'acquisition**. Nuance importante : `affiliate` reste malgré tout le canal qui génère le plus de conversions au total (surtout du réachat) — son rôle réel semble davantage être un canal de fidélisation/réachat qu'un canal d'acquisition, ce qui explique ce mauvais LTV:CAC sans remettre en cause toute sa valeur.


## Synthèse — Étape 1

- Les deux fichiers `campaigns.csv` et `touchpoints.csv` se réconcilient parfaitement — un point positif après le Jour 1.
- Le classement des canaux **change du tout au tout selon la métrique** : volume, ROAS, CPA/CAC et LTV:CAC racontent chacun une histoire différente.
- `display` et `social` totalisent 58 % du volume de points de contact mais **0 conversion mesurée** sur la colonne `converted` — un signal fort qu'il ne faut **pas** les juger sur leur conversion directe (dernier contact). C'est l'objet du notebook suivant.
- Le CAC est fragile statistiquement (1,9 % de primo-achats seulement) et doit être lu comme un ordre de grandeur.
- `affiliate` est le canal le plus coûteux et le moins bon en LTV:CAC en tant que canal d'**acquisition**, mais reste le premier contributeur en volume de conversions — probablement un rôle davantage tourné vers la fidélisation/réachat.

➡️ Suite dans `TP3 - Attribution Multicanal - Lumina & Co.ipynb`.
